# 16 · Memory Systems for Agents

Working, episodic and semantic memory, retrieval methods, and the real Mem0 library.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfzalManihar/Agentic-AI-Learning/blob/main/16_memory_systems.ipynb)

**Covered in this notebook**

- Memory Systems in Agentic AI — A Practical Demo
- 0. Setup
- 1. The core problem: an LLM forgets everything
- 2. Working memory — the context window
- 2.2 The test question
- 2.3 Method B — Naive truncation (just cut the old messages)
- 2.4 Method C — Summarisation
- Working memory — advantages and disadvantages
- Retrieval tools (used by the next three memories)
- 3. Episodic memory — "what happened before"
- 3.1 Store: summaries + key events + outcomes
- 3.2 Retrieval method 1 — Chronological

> Part of [Agentic AI Learning](https://github.com/AfzalManihar/Agentic-AI-Learning). Add your own API keys in Colab Secrets (key icon) before running.

#10-10-2026

Topic :- Memory System

1- Working memory

2- episodic memory

3- semantic memory

4- procedural memory

5- mem0 practical Agent Memory

# Memory Systems in Agentic AI — A Practical Demo

An LLM has **no memory between calls**. Every call starts from zero.
Everything an agent "remembers" is something *we* store and then put back into the prompt.

In this notebook you will see the **four memory types** working with **real Groq API calls** (no fake answers):

| # | Memory type | Question it answers | What we build |
|---|---|---|---|
| 1 | **Working memory** | What is happening *right now*? | Chat history, truncation, summarisation, selective retention |
| 2 | **Episodic memory** | What happened *before*? | Past sessions saved with a timestamp, retrieved when relevant |
| 3 | **Semantic memory** | What do I *know* (facts)? | User facts saved as key-value, retrieved with embeddings |
| 4 | **Procedural memory** | *How* do I do things? | Saved step-by-step procedures, loaded into the prompt |

**Story used everywhere:** a travel assistant for a user called **Riya**
(vegetarian, severe nut allergy, planning a 4-day Goa trip).

**Requirements:** Colab, a Groq key saved in Colab *Secrets* with the name `groq_api` (turn on "Notebook access").
Run the cells **top to bottom**.

## 0. Setup
We install two libraries, load the API key, and write **one helper function `chat()`**
that every demo uses. It also:
- counts LLM calls (`LLM_CALLS`)
- remembers the real prompt size from Groq (`LAST_PROMPT_TOKENS`)
- waits and retries when Groq says "rate limit" (free tier)

In [2]:
# Install libraries (takes ~1 minute the first time)
!pip install -q groq sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.4 MB/s eta 0:00:00


In [3]:
import os, re, json, time
from datetime import datetime

import numpy as np
import pandas as pd
from groq import Groq
from google.colab import userdata

# Load the key from Colab Secrets (name: groq_api)
client = Groq(api_key=userdata.get("groq_api"))

MODEL = "openai/gpt-oss-20b"   # one small, fast Groq model for everything

LLM_CALLS = 0            # how many real LLM calls we made
LAST_PROMPT_TOKENS = 0   # real prompt size (in tokens) of the latest call

def chat(messages, json_mode=False, max_tokens=1500):
    """Send messages to Groq and return the reply text. Retries on rate limits."""
    global LLM_CALLS, LAST_PROMPT_TOKENS
    for attempt in range(8):
        try:
            kwargs = dict(model=MODEL, messages=messages,
                          max_tokens=max_tokens, temperature=0.3)
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}
            resp = client.chat.completions.create(**kwargs)
            text = (resp.choices[0].message.content or "").strip()
            if json_mode and not text:
                raise ValueError("json_validate_failed: empty reply")
            LLM_CALLS += 1
            LAST_PROMPT_TOKENS = resp.usage.prompt_tokens
            return text
        except Exception as e:
            msg = str(e)
            if "429" in msg or "rate_limit" in msg:        # too many tokens per minute
                m = re.search(r"try again in ([\d.]+)(ms|s)", msg)
                wait = float(m.group(1)) / (1000 if m and m.group(2) == "ms" else 1) if m else 5
                time.sleep(wait + 1)
            elif "json_validate_failed" in msg or "empty reply" in msg:
                time.sleep(1)                               # JSON mode sometimes fails once
            else:
                raise                                       # real error (bad key etc.)
    raise RuntimeError("Groq did not answer after several retries")

def ask_json(messages):
    """Like chat(), but returns a Python dict (the model must reply in JSON)."""
    for _ in range(3):
        text = chat(messages, json_mode=True)
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass
    raise RuntimeError("Model did not return valid JSON")

def user(text):      return {"role": "user", "content": text}
def assistant(text): return {"role": "assistant", "content": text}
def system(text):    return {"role": "system", "content": text}

# Start clean: delete memory files from older runs
for f in ["episodic_memory.json", "semantic_memory.json", "procedural_memory.json"]:
    if os.path.exists(f):
        os.remove(f)

# Quick test that the key and model work
print(chat([user("Say 'ready' in one word.")]))

ready


## 1. The core problem: an LLM forgets everything

We make **two separate calls**. In call 1 we tell the model about Riya.
In call 2 we ask a question that needs that information — but we send nothing from call 1.

In [4]:
# Call 1: give the model some personal information
r1 = chat([user("Hi, I'm Riya. I'm vegetarian and allergic to nuts. I'm planning a 4-day Goa trip in December.")])
print("CALL 1 reply:\n", r1)

# Call 2: a brand-new call. We do NOT send call 1 again.
r2 = chat([user("What should I avoid eating on my trip?")])
print("\nCALL 2 reply:\n", r2)

CALL 1 reply:
 Hi Riya! 🌞  
A 4‑day December getaway to Goa sounds amazing, and I’m happy to help you plan a nut‑free, vegetarian‑friendly itinerary. Below is a mix of beach time, culture, food, and a few practical tips for traveling with a nut allergy. Feel free to tweak it to match your budget, pace, or any other preferences you have.

---

## Quick Overview

| Day | Focus | Highlights | Vegetarian & Nut‑Free Food Ideas |
|-----|-------|------------|----------------------------------|
| 1 | Arrival & South Goa | Palolem, Agonda, quiet vibe | **Mum’s Kitchen** (Panaji) – “Goan vegetarian curry” |
| 2 | North Goa Beaches & Markets | Baga, Calangute, Anjuna Flea Market | **The Fisherman’s Wharf** (no nuts) – “Vegetable Xacuti” |
| 3 | Culture & Spice | Old Goa, Basilica, Spice Plantation | **Vinayak’s** (Panaji) – “Mushroom Masala” |
| 4 | Relax & Departure | Sunset at Chapora, last‑minute shopping | **The Fisherman’s Wharf** (no nuts) – “Paneer Tikka” |

> **Tip:** December is peak sea

**What you should see:** in call 2 the model does *not* know Riya is vegetarian or allergic to nuts.
It gives generic advice (or asks who you are). The model keeps nothing between calls.

> Every memory system below is just a way of **choosing what to put back into the prompt**.

## 2. Working memory — the context window

Working memory = everything inside the prompt right now (system prompt + chat history + tool results).
The model can *only* use what is in this list.

### 2.1 Build a real conversation
We keep a `history` list and send the **whole list** on every call.
Watch `prompt tokens` grow every turn — this is the real number from Groq.

In [5]:
SYSTEM = "You are a friendly travel assistant. Answer in at most 3 short sentences."

user_turns = [
    "Hi, I'm Riya. I'm vegetarian and severely allergic to nuts. Please never suggest non-veg food or anything with nuts.",
    "I'm planning a 4-day Goa trip in December with a budget of 40000 INR.",
    "Which beaches are good for a quiet morning walk?",
    "What is the weather usually like in December?",
    "How should I get around in Goa?",
    "What should I pack for the trip?",
]

history = [system(SYSTEM)]          # working memory = this list

for text in user_turns:
    history.append(user(text))
    reply = chat(history)           # we send the WHOLE history every time
    history.append(assistant(reply))
    print(f"Turn {len(history)//2}: prompt tokens = {LAST_PROMPT_TOKENS}")

print("\nMessages in working memory:", len(history))

Turn 1: prompt tokens = 116
Turn 2: prompt tokens = 187
Turn 3: prompt tokens = 345
Turn 4: prompt tokens = 436
Turn 5: prompt tokens = 513
Turn 6: prompt tokens = 615

Messages in working memory: 13


### 2.2 The test question
Now we ask something where the early constraints matter (vegetarian + nut allergy).
We will answer it **four ways** and compare.

A small "judge" call checks each answer against Riya's rules.

In [6]:
FINAL_Q = "Suggest one dinner for me tonight."

def respects_rules(reply):
    """Ask the LLM to check if a reply follows Riya's rules. Returns YES or NO."""
    verdict = chat([
        system("You are a strict checker. Rules: the food must be 100% vegetarian AND contain no nuts. "
               "Reply with exactly one word: YES if the answer follows both rules, otherwise NO."),
        user("Answer to check:\n" + reply),
    ])
    return "YES" if verdict.strip().upper().startswith("YES") else "NO"

results = []   # we collect one row per method for a final table

def run_method(name, messages):
    """Ask the final question with the given messages and record the result."""
    reply = chat(messages + [user(FINAL_Q)])
    tokens = LAST_PROMPT_TOKENS
    ok = respects_rules(reply)
    results.append({"method": name, "prompt tokens": tokens, "follows Riya's rules?": ok})
    print(f"--- {name} ---\nPrompt tokens: {tokens} | Follows rules: {ok}\nReply: {reply}\n")

# Method A: send the FULL history (best memory, biggest prompt)
run_method("A. Full history", history)

--- A. Full history ---
Prompt tokens: 698 | Follows rules: YES
Reply: Try a quick vegetable stir‑fry with tofu, bell peppers, broccoli, and a light soy‑ginger glaze. Serve it over steamed rice or quinoa for a filling meal. Finish with a cool cucumber raita to keep things refreshing.



### 2.3 Method B — Naive truncation (just cut the old messages)
We keep only the system prompt + the **last 4 messages**.
The first message (the allergy!) is thrown away.

In [7]:
RECENT = 4   # how many latest messages we always keep

truncated = [history[0]] + history[-RECENT:]
run_method("B. Truncation (last 4 only)", truncated)

--- B. Truncation (last 4 only) ---
Prompt tokens: 283 | Follows rules: NO
Reply: Try a spicy Goan fish curry with steamed rice and a side of sautéed greens. Finish with a fresh coconut water to keep you hydrated.



**Look at the reply.** The allergy and the vegetarian rule are gone from the prompt,
so the model can suggest anything. This is the danger from your notes:
*truncating can delete the user's most important constraint.*

LLM answers vary from run to run. If method B happens to answer safely, that is luck, not memory — the model had no way to know the rule.

### 2.4 Method C — Summarisation
We ask the LLM to compress the **old** messages into a short summary and
tell it to keep every constraint. The summary replaces the raw messages.

In [8]:
old_messages = history[1:-RECENT]      # everything except system prompt and last 4

def to_text(msgs):
    return "\n".join(f"{m['role']}: {m['content']}" for m in msgs)

summary = chat([
    system("You compress conversations. Write a summary in under 80 words. "
           "ALWAYS keep every user constraint, allergy, preference, number and decision exactly."),
    user("Conversation:\n" + to_text(old_messages)),
])
print("SUMMARY:\n", summary, "\n")

# New working memory = system prompt + summary + last 4 messages
summarised = [system(SYSTEM + "\n\nSummary of the earlier conversation:\n" + summary)] + history[-RECENT:]
run_method("C. Summarisation", summarised)

SUMMARY:
 Riya, vegetarian, nut‑allergic, wants a 4‑day Goa trip in December on ₹40,000. Assistant recommends budget guesthouse (₹1,200–₹1,800/night), scooter (₹1,200/day), nut‑free veg meals at The Green House/Mango Tree Café, beach yoga, spice‑farm tour, sunset cruise (₹1,500–₹2,000). Quiet morning walks suggested at Palolem, Mandrem, Miramar. Weather: mild 20–28 °C, clear skies, low humidity. Assistant will avoid non‑veg or nuts. 

--- C. Summarisation ---
Prompt tokens: 414 | Follows rules: YES
Reply: Try a classic **Chana Masala** (spiced chickpea curry) served with warm roti or basmati rice.  
Add a side of cucumber raita (use plain yogurt, no nuts) and a fresh green salad.  
It’s hearty, 100 % vegetarian, and completely nut‑free.



:### 2.5 Method D — Selective retention
Instead of summarising, we **score each old message** for how important it is
for the *current question* (0 = useless, 10 = essential) and keep only the high scorers.

In [9]:
numbered = "\n".join(f"{i} [{m['role']}]: {m['content'][:300]}" for i, m in enumerate(old_messages))

scores = ask_json([
    system("You rate conversation messages. For the given question, give each message an importance "
           "score from 0 to 10 (10 = answer would be wrong or unsafe without it). "
           "Reply in JSON like {\"scores\": [{\"id\": 0, \"score\": 7}]} with one entry per message."),
    user(f"Question: {FINAL_Q}\n\nMessages:\n{numbered}"),
])["scores"]

score_by_id = {int(s["id"]): s["score"] for s in scores}
THRESHOLD = 6
keep_ids = [i for i in range(len(old_messages)) if score_by_id.get(i, 10) >= THRESHOLD]  # unknown -> keep

for i, m in enumerate(old_messages):
    flag = "KEEP" if i in keep_ids else "drop"
    print(f"{flag} (score {score_by_id.get(i, '?')}) [{m['role']}] {m['content'][:70]}")

selected = [history[0]] + [old_messages[i] for i in keep_ids] + history[-RECENT:]
print()
run_method("D. Selective retention", selected)

KEEP (score 10) [user] Hi, I'm Riya. I'm vegetarian and severely allergic to nuts. Please nev
drop (score 2) [assistant] Hi Riya! I’ll keep your vegetarian and nut‑allergy preferences in mind
drop (score 1) [user] I'm planning a 4-day Goa trip in December with a budget of 40000 INR.
KEEP (score 7) [assistant] Here’s a quick 4‑day plan for Goa on ₹40,000: stay in a budget-friendl
drop (score 1) [user] Which beaches are good for a quiet morning walk?
drop (score 1) [assistant] For a peaceful morning stroll, try Palolem Beach in South Goa – its ca
drop (score 1) [user] What is the weather usually like in December?
drop (score 1) [assistant] In December, Goa enjoys mild, pleasant weather with temperatures rangi

--- D. Selective retention ---
Prompt tokens: 457 | Follows rules: YES
Reply: Try a **Paneer Tikka Masala**: marinated paneer cubes grilled and simmered in a tomato‑based sauce with spices, served with basmati rice or whole‑wheat naan (no nuts). It’s hearty, vegetarian, and nut‑fre

In [10]:
# Compare all four methods using real numbers
pd.DataFrame(results)

,method,prompt tokens,follows Riya's rules?
0,A. Full history,698,YES
1,B. Truncation (last 4 only),283,NO
2,C. Summarisation,414,YES
3,D. Selective retention,457,YES


### Working memory — advantages and disadvantages

| Method | Advantages | Disadvantages |
|---|---|---|
| **Full history** | Nothing is lost; simplest | Prompt grows every turn; cost and latency rise; hits the context limit |
| **Truncation** | Very simple; fixed size | Can delete critical constraints (unsafe) |
| **Summarisation** | Keeps key facts in far fewer tokens | Costs one extra LLM call; details can be lost if the summary is poor |
| **Selective retention** | Keeps only what matters for the current question | Needs a scoring call each time; a wrong score drops something important |

**Prompt caching** (from your notes) is a separate trick: static parts such as the system prompt are cached by the provider so they are not reprocessed on every call.

## Retrieval tools (used by the next three memories)
Episodic, semantic and procedural memory all need **retrieval**: finding the few memories that match the current question.
We use **embeddings** — a small model turns text into numbers so that *similar meaning = similar numbers*.
We then rank memories by **cosine similarity**.

In [11]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")   # small, free, runs inside Colab

def embed(texts):
    return embedder.encode(texts, normalize_embeddings=True)   # unit vectors

def top_k(query, texts, k=2):
    """Return [(index, similarity), ...] of the k texts most similar to the query."""
    if not texts:
        return []
    sims = embed(texts) @ embed([query])[0]          # cosine similarity (vectors are normalised)
    best = np.argsort(-sims)[:k]
    return [(int(i), float(sims[i])) for i in best]

# Tiny test: the food sentence should score highest for a food question
demo = ["Riya is vegetarian", "The flight leaves at 6 am", "Goa is hot in May"]
for i, s in top_k("What can she eat?", demo, k=3):
    print(f"{s:.2f}  {demo[i]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.39  Riya is vegetarian
0.10  Goa is hot in May
-0.00  The flight leaves at 6 am


## 3. Episodic memory — "what happened before"

Episodic memory is a **persistent store of past events, conversations and interactions**:
*what happened, when, with whom.*

**Why agents need it:** a user returns after a week and says *"can you continue what we were doing?"*
Without episodic memory the agent has no idea what they were doing.

**What gets stored** (from your notes):
1. **Conversation summaries** — not raw transcripts (too long), but a compressed version: what was discussed, decided and produced.
2. **Key events** — e.g. "user approved the plan", "user prefers concise answers".
3. **Outcomes** — what action was taken and what happened as a result.

**How retrieval works:**
- **Chronological** — "give me the last N sessions"
- **Semantic** — "find past sessions similar to the current task"

First, let us see the problem for real.

In [12]:
# Cell 2
# The user comes back after a week. The agent has NO episodic memory yet.
COMEBACK = "Hi, I'm back after a week. Can you continue what we were doing?"

no_memory_reply = chat([system(SYSTEM), user(COMEBACK)])
print("WITHOUT episodic memory:\n", no_memory_reply)

WITHOUT episodic memory:
 Sure! What were we working on? Let me know the details and we can pick up from there.


### 3.1 Store: summaries + key events + outcomes
After every session the LLM writes **one episode record** with three parts:

| Field | What it holds |
|---|---|
| `summary` | 2-3 sentences: discussed, decided, produced, and what is still pending |
| `key_events` | short list of important moments |
| `outcomes` | action taken -> what happened |

The record is saved to a JSON file **with a timestamp**.

> **Note:** in this demo all sessions run within minutes. To make "a week later" realistic,
> we pass `days_ago` so each record gets a *back-dated* timestamp. This is for the demo only; in a real app the time is simply "now".

In [13]:
# Cell 4
from datetime import timedelta

EPISODE_FILE = "episodic_memory.json"

def load_episodes():
    """Read all saved episodes from the file."""
    if not os.path.exists(EPISODE_FILE):
        return []
    with open(EPISODE_FILE) as f:
        return json.load(f)

def run_session(user_lines):
    """Have a real conversation and return the message list."""
    h = [system(SYSTEM)]
    for line in user_lines:
        h.append(user(line))
        h.append(assistant(chat(h)))
    return h

def save_episode(h, days_ago=0):
    """Compress a finished conversation into ONE episode record and save it."""
    record = ask_json([
        system("You write memory records of conversations. Reply in JSON with exactly these keys: "
               "summary (2-3 sentences: what was discussed, decided and produced, and what is still pending), "
               "key_events (a list of short strings, e.g. 'user approved the plan'), "
               "outcomes (a list of short strings: action taken and what happened as a result). "
               "Do not copy the conversation - compress it."),
        user("Conversation:\n" + to_text(h[1:]) + "\n\nWrite the JSON record."),
    ])
    # Clean the record so later code can rely on its shape
    record["summary"] = str(record.get("summary", ""))
    for key in ("key_events", "outcomes"):
        value = record.get(key, [])
        record[key] = value if isinstance(value, list) else [str(value)]
    record["time"] = (datetime.now() - timedelta(days=days_ago)).strftime("%Y-%m-%d %H:%M")
    record["raw_chars"] = len(to_text(h[1:]))          # size of the transcript we did NOT store

    episodes = load_episodes()
    episodes.append(record)
    with open(EPISODE_FILE, "w") as f:
        json.dump(episodes, f, indent=2)
    return record

def show_episode(e):
    print(f"[{e['time']}]  (summary {len(e['summary'])} chars vs transcript {e['raw_chars']} chars)")
    print("  Summary   :", e["summary"])
    print("  Key events:", e["key_events"])
    print("  Outcomes  :", e["outcomes"], "\n")

# Four real sessions. days_ago makes the timestamps look like a real week.
sessions = [
    (7, ["Plan a 4-day Goa itinerary for me within 40000 INR.",
         "I approve this plan. Please keep your answers short and to the point from now on."]),
    (5, ["Suggest a beach hotel in Goa for around 5000 INR per night.",
         "That sounds too crowded and loud for me. I prefer quiet places away from the party area."]),
    (3, ["Suggest a vegetarian restaurant in Goa.",
         "I tried the place you suggested and loved it, thanks!"]),
    (1, ["Let's work on day 2 of the Goa plan. Suggest a morning activity.",
         "Good, add the sunrise walk at Palolem to day 2. I will decide the evening plan tomorrow."]),
]

if os.path.exists(EPISODE_FILE):          # safe to re-run this cell
    os.remove(EPISODE_FILE)

for days_ago, lines in sessions:
    episode = save_episode(run_session(lines), days_ago=days_ago)
    show_episode(episode)

[2026-10-04 19:37]  (summary 175 chars vs transcript 621 chars)
  Summary   : User asked for a 4-day Goa itinerary under 40000 INR. Assistant provided a detailed plan and user approved it, requesting concise future responses. No further actions pending.
  Key events: ['user requested Goa itinerary', 'assistant provided 4-day plan', 'user approved plan', 'user requested concise answers']
  Outcomes  : ['assistant delivered itinerary', 'assistant acknowledged approval', 'assistant committed to concise responses'] 

[2026-10-06 19:37]  (summary 263 chars vs transcript 766 chars)
  Summary   : The user asked for a beach hotel in Goa around 5000 INR per night. The assistant first recommended Baga Beach Resort, but the user found it too crowded and requested a quieter option. The assistant then suggested Palolem Beach Resort, a quieter spot in South Goa.
  Key events: ['user requested budget beach hotel', 'assistant suggested Baga Beach Resort', 'user rejected due to crowding', 'assistant su

**What you should see:** each session became a short record. The summary is much smaller than the raw transcript
(compare the character counts), and the key events / outcomes hold the details that matter
(approved plan, "prefers concise answers", "quiet hotel preferred", "loved the restaurant", "evening of day 2 still open").

### 3.2 Retrieval method 1 — Chronological
"Give me the last N sessions." We simply **sort by timestamp** and take the newest ones. No embeddings needed.
Best for: *"continue what we were doing"*, *"what happened last time?"*

In [14]:
# Cell 7
def days_since(e):
    """How many days ago did this episode happen?"""
    return (datetime.now() - datetime.strptime(e["time"], "%Y-%m-%d %H:%M")).days

def episode_text(e):
    """Turn one episode record into text we can put in a prompt."""
    return (f"[{e['time']}, {days_since(e)} days ago] Summary: {e['summary']} "
            f"Key events: {'; '.join(e['key_events'])}. "
            f"Outcomes: {'; '.join(e['outcomes'])}.")

def last_sessions(n=3):
    """CHRONOLOGICAL retrieval: the n newest episodes, oldest first."""
    episodes = sorted(load_episodes(), key=lambda e: e["time"])
    return episodes[-n:]

for e in last_sessions(2):
    print(episode_text(e), "\n")

[2026-10-08 19:37, 3 days ago] Summary: User asked for vegetarian restaurants in Goa; assistant suggested The Little Green and Moksha. User reported enjoying the recommendation and thanked the assistant. Assistant thanked and offered another option, The Green House. Key events: user requested vegetarian restaurant suggestions; assistant suggested The Little Green and Moksha; user confirmed they liked the suggestion; assistant thanked user and suggested The Green House. Outcomes: assistant provided restaurant recommendations; user enjoyed the recommended restaurant. 

[2026-10-10 19:37, 1 days ago] Summary: User requested a morning activity for day 2 of the Goa plan. Assistant suggested several options and the user decided to add a sunrise walk at Palolem. The evening plan remains to be decided tomorrow. Key events: user requested morning activity for day 2; assistant suggested sunrise stroll, kayak tour, spice plantation; user added sunrise walk at Palolem; assistant confirmed addition

### 3.3 Retrieval method 2 — Semantic
"Find past sessions similar to the current task." We embed each episode and rank by **cosine similarity**
(same `top_k` helper as before). Best for: *"recommend a hotel"* -> find the session about hotels, even if it was long ago.

Below we ask the **same question** with both methods and compare.

In [15]:
# Cell 9
def retrieve_episodes(query, k=2):
    """SEMANTIC retrieval: the k episodes most similar to the query, as (text, similarity)."""
    episodes = load_episodes()                       # read from the file every time
    texts = [f"{e['summary']} {' '.join(e['key_events'])} {' '.join(e['outcomes'])}" for e in episodes]
    return [(episode_text(episodes[i]), s) for i, s in top_k(query, texts, k)]

QUESTION = "Can you recommend a hotel for my next Goa trip?"
MEMORY_SYSTEM = SYSTEM + "\n\nPast sessions with this user:\n"

# Method 1: chronological -> just the 2 newest sessions
chrono = [episode_text(e) for e in last_sessions(2)]
print("CHRONOLOGICAL picked:")
for t in chrono:
    print("  -", t[:110], "...")
print("\nAnswer:\n", chat([system(MEMORY_SYSTEM + "\n".join("- " + t for t in chrono)), user(QUESTION)]))

# Method 2: semantic -> the 2 sessions closest to the question
semantic = retrieve_episodes(QUESTION, k=2)
print("\nSEMANTIC picked:")
for t, sim in semantic:
    print(f"  - (similarity {sim:.2f})", t[:110], "...")
print("\nAnswer:\n", chat([system(MEMORY_SYSTEM + "\n".join("- " + t for t, _ in semantic)), user(QUESTION)]))

CHRONOLOGICAL picked:
  - [2026-10-08 19:37, 3 days ago] Summary: User asked for vegetarian restaurants in Goa; assistant suggested The  ...
  - [2026-10-10 19:37, 1 days ago] Summary: User requested a morning activity for day 2 of the Goa plan. Assistant ...

Answer:
 Sure! For a relaxed stay near Palolem, try **The Palolem Beach Resort** – it’s a boutique hotel with a great vegetarian café and easy beach access. If you prefer a bit more luxury, **The Leela Goa** offers spacious rooms, a spa, and a dedicated vegetarian menu. Both are highly rated and close to the activities you’ve planned.

SEMANTIC picked:
  - (similarity 0.54) [2026-10-06 19:37, 5 days ago] Summary: The user asked for a beach hotel in Goa around 5000 INR per night. The ...
  - (similarity 0.41) [2026-10-04 19:37, 7 days ago] Summary: User asked for a 4-day Goa itinerary under 40000 INR. Assistant provid ...

Answer:
 Try **Agonda Beach Resort** in South Goa – it’s a quiet, family‑friendly spot with rooms around ₹4,5

**What you should see:**
- *Chronological* picked the two newest sessions (restaurant and day 2). The hotel session is too old, so the answer does not use the "quiet place" lesson.
- *Semantic* picked the **hotel session** (highest similarity), so the answer suggests a quiet place.

**Rule of thumb:** "continue / last time" -> chronological. "Something similar to this task" -> semantic.

### 3.4 Putting it to work: "continue what we were doing"
Now the problem from the start of this section. We fetch the **last 3 sessions chronologically**
and ask the same question again. Compare with the no-memory answer from the first cell.

In [16]:
# Cell 12
recent = last_sessions(3)
print("Memory given to the agent:")
for e in recent:
    print("  -", episode_text(e)[:120], "...")

CONTINUE_SYSTEM = ("You are a friendly travel assistant. Answer in at most 5 short sentences.\n\n"
                   "The user is returning. Their most recent sessions (oldest first):\n"
                   + "\n".join("- " + episode_text(e) for e in recent)
                   + "\n\nContinue from where they left off and mention anything still pending.")

with_memory_reply = chat([system(CONTINUE_SYSTEM), user(COMEBACK)])

print("\nWITHOUT episodic memory:\n", no_memory_reply)
print("\nWITH episodic memory:\n", with_memory_reply)

Memory given to the agent:
  - [2026-10-06 19:37, 5 days ago] Summary: The user asked for a beach hotel in Goa around 5000 INR per night. The assistant ...
  - [2026-10-08 19:37, 3 days ago] Summary: User asked for vegetarian restaurants in Goa; assistant suggested The Little Gre ...
  - [2026-10-10 19:37, 1 days ago] Summary: User requested a morning activity for day 2 of the Goa plan. Assistant suggested ...

WITHOUT episodic memory:
 Sure! What were we working on? Let me know the details and we can pick up from there.

WITH episodic memory:
 Welcome back! We added a sunrise walk at Palolem for Day 2. The evening plan for Day 2 is still open—would you like a beachside dinner, a cultural show, or a quiet walk? Also, we haven’t set Day 3 activities yet; let me know what you’d like to do next.


**What you should see:** without memory the agent asks "what were we doing?". With memory it knows the Goa plan,
that day 2 has the sunrise walk, and that the **evening plan is still pending**.

### Episodic memory — advantages and disadvantages

| Advantages | Disadvantages |
|---|---|
| Agent can **continue** work after days or weeks | Grows forever - needs pruning or archiving |
| Summaries are small, so many sessions fit in the prompt | A weak summary **loses details** (the raw transcript is gone) |
| Key events and outcomes help the agent learn what worked | Extra LLM call to write every record |
| Two retrieval modes: by time and by meaning | Chronological misses old but relevant sessions; semantic can pick the wrong one |



## 4. Semantic memory — two kinds of knowledge

| Kind | What it is | Example | Retrieval |
|---|---|---|---|
| **World knowledge** | Same for everybody | "Machine learning is done in Python" | Classic RAG (chunk, embed, retrieve) |
| **User knowledge** | Only about *this* user | "The user does not know Python" | RAG is **not enough** (shown next) |

In [17]:
# Cell S2 | WORLD knowledge = classic RAG (same facts for every user)
WORLD_DOCS = [
    "Machine learning work is mostly done in Python, using libraries such as PyTorch and scikit-learn.",
    "Transformers are neural networks that use attention; the Hugging Face library is the common way to use them.",
    "Goa has its peak tourist season from November to February.",
    "Pune is a city in Maharashtra known for IT companies and education.",
    "Paneer is a vegetarian protein source common in Indian cooking.",
]

QUESTION = "How are Transformers usually used in code?"

# Retrieve the 2 most similar documents (same top_k helper as before)
found = top_k(QUESTION, WORLD_DOCS, k=2)
for i, sim in found:
    print(f"retrieved (similarity {sim:.2f}): {WORLD_DOCS[i]}")

context = "\n".join("- " + WORLD_DOCS[i] for i, _ in found)
print("\nAnswer:\n", chat([system("Answer in at most 3 short sentences using only this context:\n" + context),
                          user(QUESTION)]))

retrieved (similarity 0.64): Transformers are neural networks that use attention; the Hugging Face library is the common way to use them.
retrieved (similarity 0.26): Machine learning work is mostly done in Python, using libraries such as PyTorch and scikit-learn.

Answer:
 Transformers are neural networks that use attention. In code they are typically accessed through the Hugging Face library. Most of this work is done in Python, often with PyTorch or scikit‑learn.


### 4b.1 Why RAG fails for USER knowledge
Query: *"Give me code for Transformers."*
The fact **"the user does not know Python"** is very important for a good answer,
but the query never mentions Python or the user's skills. Similarity search may skip it.

In [18]:
# Cell S3 | Plain similarity search (RAG) on USER facts
USER_FACTS = [
    "The user does not know Python.",
    "The user is studying AI and machine learning.",
    "The user prefers short answers.",
    "The user's deadline is Friday.",
    "The user is vegetarian.",
    "The user lives in Pune.",
]

QUESTION = "Give me code for Transformers."
IMPORTANT = "The user does not know Python."      # the fact a good answer MUST use

found = top_k(QUESTION, USER_FACTS, k=2)           # similarity only, top 2
retrieved = [USER_FACTS[i] for i, _ in found]
for i, sim in found:
    print(f"retrieved (similarity {sim:.2f}): {USER_FACTS[i]}")
print("\nDid RAG retrieve the important fact?", IMPORTANT in retrieved)

# Answer using ONLY the retrieved facts
facts_text = "\n".join("- " + f for f in retrieved)
print("\nAnswer with RAG-retrieved facts:\n",
      chat([system("You are a coding tutor. Answer briefly.\n\nKnown facts about the user:\n" + facts_text),
            user(QUESTION)]))

retrieved (similarity 0.14): The user's deadline is Friday.
retrieved (similarity 0.14): The user is studying AI and machine learning.

Did RAG retrieve the important fact? False

Answer with RAG-retrieved facts:
 Below is a **minimal, self‑contained PyTorch implementation of a Transformer encoder block** (attention + feed‑forward) that you can drop into a larger model.  
After that you’ll see a one‑liner example of how to fine‑tune a pre‑trained model from Hugging Face – the quickest way to get a working transformer for most tasks.

---

## 1.  Tiny Transformer Encoder Block (PyTorch)

```python
import torch
import torch.nn as nn
import math

class TinyTransformerEncoder(nn.Module):
    def __init__(self, d_model=64, nhead=4, dim_ff=256, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)

        self.ffn = nn.Sequential(
        

## 4c. Semantic memory with the real Mem0 library
Now the same idea with the actual **`mem0ai`** library. Mem0 uses three parts:
- **LLM** (Groq) to extract facts and decide ADD / UPDATE / DELETE
- **Embedder** (local HuggingFace model) to turn memories into vectors
- **Vector store** (local Qdrant, saved in a folder) to keep the memories

In [ ]:
# Cell M2
!pip uninstall -y mem0ai
!pip install -q mem0ai==1.0.11
import os
os.kill(os.getpid(), 9)   # restarts the runtime so the new version loads

Found existing installation: mem0ai 1.0.11
Uninstalling mem0ai-1.0.11:
  Successfully uninstalled mem0ai-1.0.11


In [1]:
import mem0
print(mem0.__version__)   # must print 1.0.11

1.0.11


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [2]:
# Cell M3 (all-in-one) | Run ONCE after the restart. Do NOT run the pip cell again.
import os, re, time, shutil
from google.colab import userdata
import mem0
print("mem0 version:", mem0.__version__)          # must be 1.0.11
from mem0 import Memory
from groq import Groq

assert "m" not in globals(), "Mem0 already created. Restart session and run this cell ONCE."

os.environ["GROQ_API_KEY"] = userdata.get("groq_api")
os.environ["MEM0_TELEMETRY"] = "False"
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-20b"

# Small helpers used by the later cells
def system(t):    return {"role": "system", "content": t}
def user(t):      return {"role": "user", "content": t}
def assistant(t): return {"role": "assistant", "content": t}

def chat(messages, max_tokens=1000):
    for _ in range(6):
        try:
            r = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=max_tokens, temperature=0.3)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e):
                time.sleep(10)
            else:
                raise
    raise RuntimeError("Groq did not answer")

# Mem0 config: Groq LLM + free local embedder + local vector store
shutil.rmtree("/content/mem0_qdrant", ignore_errors=True)
config = {
    "llm": {"provider": "groq",
            "config": {"model": MODEL, "temperature": 0.1, "max_tokens": 1000}},
    "embedder": {"provider": "huggingface", "config": {"model": "all-MiniLM-L6-v2"}},
    "vector_store": {"provider": "qdrant",
                     "config": {"collection_name": "riya_memories", "path": "/content/mem0_qdrant",
                                "embedding_model_dims": 384, "on_disk": True}},
}
m = Memory.from_config(config)
USER = "riya"
print("Mem0 ready")

mem0 version: 1.0.11


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/mem0/embeddings/huggingface.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.config.embedding_dims = self.config.embedding_dims or self.model.get_sentence_embedding_dimension()


Mem0 ready


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 4c.1 Add — extract and store memories from a conversation
We pass a normal conversation. Mem0's LLM extracts the lasting facts and stores them.

In [3]:
# Cell M4 | ADD: Mem0 extracts facts from the conversation by itself
conversation = [
    {"role": "user", "content": "Hi, I'm Riya from Pune. I'm vegetarian and allergic to nuts."},
    {"role": "assistant", "content": "Nice to meet you Riya! I'll remember that."},
    {"role": "user", "content": "I'm studying machine learning but I don't know Python at all."},
]

result = m.add(conversation, user_id=USER)
for r in result.get("results", []):
    print(r.get("event"), "->", r.get("memory"))

ADD -> Name is Riya
ADD -> Location is Pune
ADD -> Vegetarian
ADD -> Allergic to nuts
ADD -> Studying machine learning
ADD -> Doesn't know Python


### 4c.2 Search — get the memories for the current query
`search()` returns the closest memories with a score. We also print `get_all()` to see everything stored.

In [4]:
# Cell M5 | SEARCH
QUERY = "Give me code for Transformers."

hits = m.search(QUERY, user_id=USER, limit=3)
print("SEARCH results for:", QUERY)
for h in hits.get("results", []):
    print(f"  score {h['score']:.2f} | {h['memory']}")

print("\nALL stored memories:")
for h in m.get_all(user_id=USER).get("results", []):
    print("  -", h["memory"])

SEARCH results for: Give me code for Transformers.
  score 0.21 | Doesn't know Python
  score 0.14 | Studying machine learning
  score 0.08 | Name is Riya

ALL stored memories:
  - Allergic to nuts
  - Studying machine learning
  - Doesn't know Python
  - Name is Riya
  - Vegetarian
  - Location is Pune


In [5]:
# Cell M6 | UPDATE (by the LLM) and DELETE (by id)
result = m.add([{"role": "user", "content": "Good news, I have now learned the basics of Python."}], user_id=USER)
for r in result.get("results", []):
    print(r.get("event"), "->", r.get("memory"))

# Manual delete: remove the memory that mentions Pune
for h in m.get_all(user_id=USER).get("results", []):
    if "Pune" in h["memory"]:
        m.delete(h["id"])
        print("\nDELETED:", h["memory"])

print("\nSTORED NOW:")
for h in m.get_all(user_id=USER).get("results", []):
    print("  -", h["memory"])

DELETE -> Doesn't know Python
ADD -> Learned basics of Python

DELETED: Location is Pune

STORED NOW:
  - Allergic to nuts
  - Studying machine learning
  - Learned basics of Python
  - Name is Riya
  - Vegetarian


In [6]:
# Cell M7 | Session flow: search at the START, add at the END
def mem0_session(user_message):
    found = m.search(user_message, user_id=USER, limit=5)          # START: load memories
    memories = [h["memory"] for h in found.get("results", [])]
    print("Memories loaded:", memories)

    prompt = "You are a helpful assistant. Answer briefly.\n\nKnown facts about the user:\n" + \
             "\n".join("- " + t for t in memories)
    reply = chat([system(prompt), user(user_message)])
    print("Answer:", reply)

    m.add([user(user_message), assistant(reply)], user_id=USER)    # END: store new facts

mem0_session("Suggest one dinner for tonight.")

Memories loaded: ['Vegetarian', 'Name is Riya', 'Studying machine learning', 'Learned basics of Python', 'Allergic to nuts']
Answer: **Tonight’s dinner:**  
**Vegetable Stir‑Fry with Tofu & Brown Rice**  
- Sauté bell peppers, broccoli, carrots, and snap peas in a little sesame oil.  
- Add cubed tofu, garlic, ginger, and a splash of soy sauce (or tamari).  
- Serve over steamed brown rice and sprinkle with chopped green onions.  

It’s quick, nut‑free, and packed with protein—perfect for a busy ML study night!


### 4c.5 Memories that expire
"The deadline is Friday" matters now but not next week. Mem0 supports an `expiration_date`.
After that date the memory is hidden from search.

In [7]:
# Cell M8 | Memories that expire: store the date in metadata, hide expired ones ourselves
from datetime import date

m.add([{"role": "user", "content": "My project deadline is this Friday."}],
      user_id=USER, metadata={"expires": "2026-10-16"})        # still valid
m.add([{"role": "user", "content": "My exam is on 1 October."}],
      user_id=USER, metadata={"expires": "2026-10-01"})        # already expired

def active_memories():
    """Return only memories that have not expired yet."""
    out = []
    for h in m.get_all(user_id=USER).get("results", []):
        expires = (h.get("metadata") or {}).get("expires")
        if expires and date.fromisoformat(expires) < date.today():
            continue                                           # expired -> hide
        out.append(h["memory"])
    return out

print("ACTIVE memories:")
for t in active_memories():
    print("  -", t)

ACTIVE memories:
  - Allergic to nuts
  - Studying machine learning
  - Project deadline is this Friday
  - Learned basics of Python
  - Name is Riya
  - Vegetarian


In [8]:
for h in m.get_all(user_id=USER).get("results", []):
    print(h["memory"], "|", h.get("metadata"))

Allergic to nuts | None
Studying machine learning | None
Project deadline is this Friday | {'expires': '2026-10-16'}
Learned basics of Python | None
Name is Riya | None
Exam on 1 October | {'expires': '2026-10-01'}
Vegetarian | None
